# 14 — The Complete RAG <-> CAV Loop

Closes the loop described in
[docs/interpretability-methods-notes.md §4.1](../docs/interpretability-methods-notes.md#41-a-neurolens-rag-specific-variant-literature-derived-concept-hypotheses):

```
decoded window -> RSN attribution -> RAG retrieves literature
    -> LLM extracts a candidate concept phrase from a relevant excerpt
    -> phrase mapped to a testable concept (src/neurolens/concepts.py)
    -> CAV/TCAV fit and tested against the decoded class
    -> final synthesis integrates decode + literature + concept test
```

This is the real answer to the previously-open question ("how to turn a
retrieved concept phrase into a labeled example set without heavy manual
curation"): literature phrases are mapped, via keyword matching, onto the
concepts we can *already* build labeled examples for from existing task
labels (`hand`, `foot`, `tongue`, `right_side`, `left_side`). Not a fully
general solution to arbitrary literature claims, but a real, working v1.

Every step is real: real corpus (8 papers, including two genuine
motor-cognition papers - Ehrsson et al. 2003, Meier et al. 2008), real
local LLM (`mlx-community/Llama-3.2-3B-Instruct-4bit`), real CAV fits on
a real trained model (Experiment "Transformer, 100-subject reference
split").


In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import torch

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the NeuroLens repository root.")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurolens.data_setup import make_dataloaders
from neurolens.model_builder import TransformerDecoder
from neurolens.engine import get_device
from neurolens.interpretability import load_roi_to_network, network_roi_indices
from neurolens.retrieval import load_index, load_embedding_model, load_reranker
from neurolens.pipeline import explain_decoded_window_with_cav_loop, make_mlx_generate_fn

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "hcp_ya_s1200" / "runs"
RESULTS_DIR = PROJECT_ROOT / "results"

device = get_device()
train_loader, val_loader, test_loader, info = make_dataloaders(PROCESSED_ROOT, batch_size=64)

model = TransformerDecoder(num_classes=info["num_classes"], num_conditions=info["num_conditions"], include_hrf_head=True)
model.load_state_dict(torch.load(PROJECT_ROOT / "models" / "case1_transformer_100subj" / "best.pt", map_location=device))
model.to(device)
model.eval()

roi_labels_path = PROCESSED_ROOT / "sub-100307" / "tfMRI_MOTOR_LR" / "roi_labels.tsv"
network_indices = network_roi_indices(load_roi_to_network(roi_labels_path))

chunks, embeddings = load_index(PROJECT_ROOT / "artifacts" / "paper_index")
embedding_model = load_embedding_model()
reranker = load_reranker()
generate_fn = make_mlx_generate_fn()

print("device:", device, "| corpus chunks:", len(chunks))


/Users/srinivasgovindasurampudi/miniconda3/envs/neurolens/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15919.72it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8052.25it/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 4418.93it/s]

device: mps | corpus chunks: 879


## Run the full loop on a handful of real test windows

In [2]:
import random

random.seed(0)
test_batches = list(test_loader)
example_indices = []
for batch in test_batches:
    for i in range(batch["x"].shape[0]):
        example_indices.append((batch, i))
sampled = random.sample(example_indices, 3)

all_results = []
for batch, i in sampled:
    x0 = batch["x"][i : i + 1]
    t0 = time.time()
    result = explain_decoded_window_with_cav_loop(
        model=model, x=x0, subject_id=batch["subject_id"][i], task=batch["task"][i], run=batch["run"][i],
        class_to_condition=info["class_to_condition"], network_indices=network_indices, device=device,
        embedding_model=embedding_model, corpus_chunks=chunks, corpus_embeddings=embeddings,
        cav_train_loader=train_loader, cav_test_loader=test_loader,
        generate_fn=generate_fn, reranker=reranker, top_k=5,
    )
    elapsed = time.time() - t0
    all_results.append(result)

    print("=" * 100)
    print(f"subject={result['subject_id']} run={result['run']} decoded={result['condition']} "
          f"confidence={result['decoded']['confidence']:.2f} ({elapsed:.0f}s)")
    print()
    print("Extracted concept phrases:")
    for p in result["extracted_concept_phrases"]:
        print(f"  - \"{p['phrase']}\" (from {p['source_file']})")
    print()
    print("CAV test results (TCAV sensitivity for the DECODED class):")
    for r in result["cav_loop_results"]:
        if not r.get("matched_concepts"):
            print(f"  - \"{r['phrase']}\" -> no known concept matched")
            continue
        for concept, vals in r["results"].items():
            print(f"  - \"{r['phrase']}\" -> {concept}: TCAV={vals['tcav_score_for_decoded_class']:.2f}")
    print()
    print("FINAL SYNTHESIS:")
    print(result["final_synthesis"])
    print()


subject=106319 run=LR decoded=baseline confidence=1.00 (34s)

Extracted concept phrases:
  - "Imagery of finger movements activates central sulcus." (from ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf)

CAV test results (TCAV sensitivity for the DECODED class):
  - "Imagery of finger movements activates central sulcus." -> hand: TCAV=0.00

FINAL SYNTHESIS:
Our analysis of the decoded brain-activity result reveals that the primary contributing resting-state network is the SomMot network, which is involved in motor control and other cognitive processes. This finding is supported by the literature excerpt on graph-based network analysis of resting-state functional MRI, which is relevant to the SomMot network. However, a literature excerpt on the Human Connectome Project's approach to resting-state fMRI is unrelated to the decoded result. Furthermore, our concept-sensitivity test using Concept Activation Vectors (

subject=106319 run=RL decoded=left_hand confidence=1.00 (48s)

Extracted concept phrases:
  - "The primary motor cortex has a somatotopic map with finger area bracketed dorsally and ventrally." (from meier-et-al-2008-complex-organization-of-human-primary-motor-cortex-a-high-resolution-fmri-study.pdf)
  - ""Left hand movement is contralateral to left motor area."" (from ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf)
  - "Tongue representation is bilateral." (from ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf)

CAV test results (TCAV sensitivity for the DECODED class):
  - "The primary motor cortex has a somatotopic map with finger area bracketed dorsally and ventrally." -> hand: TCAV=1.00
  - ""Left hand movement is contralateral to left motor area."" -> hand: TCAV=1.00
  - ""Left hand movement is contralateral to left motor area."" ->

subject=102311 run=RL decoded=left_foot confidence=1.00 (45s)

Extracted concept phrases:
  - "The primary motor cortex has a somatotopic map with finger area bracketed dorsally and ventrally." (from meier-et-al-2008-complex-organization-of-human-primary-motor-cortex-a-high-resolution-fmri-study.pdf)
  - ""Left hand movement is contralateral to left motor area."" (from ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf)
  - "The motor cortex representation of the arm and hand is complex and non-traditional." (from meier-et-al-2008-complex-organization-of-human-primary-motor-cortex-a-high-resolution-fmri-study.pdf)

CAV test results (TCAV sensitivity for the DECODED class):
  - "The primary motor cortex has a somatotopic map with finger area bracketed dorsally and ventrally." -> hand: TCAV=0.00
  - ""Left hand movement is contralateral to left motor area."" -> hand: TCAV=0.00
  - ""Left hand movement is contralatera

## Save results (excluding raw prompts, keeping the interesting artifacts)

In [3]:
summary = []
for r in all_results:
    summary.append({
        "subject_id": r["subject_id"],
        "run": r["run"],
        "decoded_condition": r["condition"],
        "confidence": r["decoded"]["confidence"],
        "query_text": r["query_text"],
        "extracted_concept_phrases": r["extracted_concept_phrases"],
        "cav_loop_results": r["cav_loop_results"],
        "final_synthesis": r["final_synthesis"],
    })

results_path = RESULTS_DIR / "rag_cav_loop_examples.json"
with open(results_path, "w") as f:
    json.dump(summary, f, indent=2)
print("saved:", results_path)


saved: /Users/srinivasgovindasurampudi/Projects/neurolens-rag/results/rag_cav_loop_examples.json


## Summary

This is the completed NeuroLens-RAG Case 1 pipeline, end to end:
ROI window -> Transformer decode -> RSN attribution -> literature retrieval
(reranked) -> LLM stance labeling -> concept-phrase extraction ->
CAV/TCAV validation against the model's actual internal representation ->
final synthesis integrating all of it. Findings and caveats written up in
[docs/interpretability-methods-notes.md §4.1](../docs/interpretability-methods-notes.md#41-a-neurolens-rag-specific-variant-literature-derived-concept-hypotheses)
and [docs/case1-summary-report.md](../docs/case1-summary-report.md).
